# 11 · GSE232381 · bulk_RNA_seq · metadata

Reads the series matrix. Writes `data/run_artifacts/GSE232381/metadata.rds`.

GEO records one clinical field: active or inactive lupus nephritis (`treatment` characteristic).
**GEO records no age, sex or SLEDAI.** Whether these patients are children cannot be confirmed from
GEO. Only the GEO accession and the activity label are kept; free-text sample descriptions are not.

## Read the sample characteristics

In [1]:
source("../src/paths.R")
suppressMessages({library(GEOquery); library(Biobase)})
pd <- pData(suppressMessages(getGEO(filename = raw("GSE232381", "GSE232381_series_matrix.txt.gz"), getGPL = FALSE)))
grep(":ch1$", names(pd), value = TRUE)

[1] "cell type:ch1" "time:ch1"      "treatment:ch1"

**Explanation**
- `getGEO()` and `pData()` as in notebook 01.
- `grep(":ch1$", names(pd), value = TRUE)` lists the characteristic fields; `value = TRUE` returns their
  names rather than their positions.

## Keep the samples in the counts table

In [2]:
cnt_header <- names(read.delim(raw("GSE232381", "GSE232381_raw_counts_GRCh38.p13_NCBI.tsv.gz"), nrows = 1, check.names = FALSE))[-1]
meta <- data.frame(sample = rownames(pd),
                   ln_active = as.numeric(pd[["treatment:ch1"]] == "active LN"),
                   in_counts = rownames(pd) %in% cnt_header, row.names = rownames(pd))
table(ln = ifelse(meta$ln_active == 1, "active", "inactive"), in_counts_table = meta$in_counts)
meta <- meta[meta$in_counts, c("sample", "ln_active")]
saveRDS(meta, art("GSE232381", "metadata.rds"))

          in_counts_table
ln         FALSE TRUE
  active       2   10
  inactive     7    6

**Explanation**
- `read.delim(..., nrows = 1)` reads only the first line of the counts table; `names()` gives its column
  names, the GSM identifiers; `[-1]` drops the first column, the gene identifier.
- `ln_active` is 1 for active lupus nephritis and 0 for inactive, from the GEO field `treatment`.
  `as.numeric(TRUE)` is 1.
- `in_counts` is TRUE for the samples that appear in NCBI's counts table.
- The table counts samples by activity and by presence in the counts table; only those present are kept.
- `saveRDS()` and `art()` as in notebook 01.

**Result.** 25 samples in GEO (12 active, 13 inactive); 16 are in the counts table: 10 active,
6 inactive.

**This is not the paper's design.** Chen et al. (*Heliyon* 2024, Methods 3.5) sequenced 9 active and
9 inactive patients. GEO lists 25 samples, and NCBI's counts table holds 16. The paper's comparison
cannot be rebuilt exactly from the public data.

**What "active" means here.** The paper defines active and inactive by renal outcome: "deterioration"
or "relapse" is active, "improvement" or "no change" is inactive (Methods 3.2). It is not a SLEDAI
category, although active episodes had a higher mean SLEDAI (12.6 against 5.1, Table 2). The paper's
cohort is pediatric-onset; GEO does not say whether each sequenced patient was under 18.